# Summarization Model Evaluation

This script evaluates and compares the performance of baseline and fine-tuned 
summarization models on financial news articles using various metrics.

## Setup and Imports

In [7]:
import os
import sys
import pandas as pd
import numpy as np
import json
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from rouge_score import rouge_scorer
import nltk
from nltk.translate.bleu_score import sentence_bleu
from collections import defaultdict
import spacy
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

## !/usr/bin/env python

## -*- coding: utf-8 -*-

## Add parent directory to path for imports

In [8]:
sys.path.append(os.path.join(os.path.dirname('__file__'), ".."))
from scripts.vector_db_manager import VectorDatabaseManager

## Initialize paths

In [9]:
DB_PATH = "../data/chroma_db"
COLLECTION_NAME = "financial_articles"
FINETUNED_MODEL_DIR = "../models/finetuned_summarizer"
BASELINE_MODEL = "facebook/bart-large-cnn"
RESULTS_DIR = "../results/model_evaluation"
os.makedirs(RESULTS_DIR, exist_ok=True)

## Download necessary NLTK data

In [10]:
try:
    nltk.download('punkt')
    nltk.download('stopwords')
except:
    print("NLTK download failed, but we'll continue anyway")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/giannisalexandrou/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/giannisalexandrou/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Initialize NLP models

In [11]:
try:
    nlp = spacy.load("en_core_web_md")
except:
    print("Spacy model not found. Installing...")
    os.system("python -m spacy download en_core_web_md")
    nlp = spacy.load("en_core_web_md")

Spacy model not found. Installing...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 23.6 MB/s eta 0:00:00a 0:00:01



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


## Initialize sentence embedding model for semantic comparison

In [12]:
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
# Connect to vector database

/Users/giannisalexandrou/.local/share/virtualenvs/trading-agent-Iwjhydeb/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


### init_vector_db



In [13]:
def init_vector_db():
    """Initialize connection to the vector database."""
    vector_db = VectorDatabaseManager(
        db_path=DB_PATH,
        collection_name=COLLECTION_NAME
    )
    return vector_db

# Retrieve article data

### get_test_articles



In [14]:
def get_test_articles(vector_db, n_articles=20, stock_symbols=None):
    """Retrieve articles for evaluation from vector database."""
    if stock_symbols is None:
        stock_symbols = ["AAPL", "GOOGL", "MSFT", "AMZN", "NVDA"]  # Default symbols
    
    articles = []
    for symbol in stock_symbols:
        results = vector_db.query_database(
            query=f"latest news about {symbol}",
            n_results=n_articles // len(stock_symbols),
            include_summary=True
        )
        
        if results and 'documents' in results:
            docs = results['documents']
            if isinstance(docs[0], list):
                docs = docs[0]
            
            metadatas = results['metadatas']
            if isinstance(metadatas[0], list):
                metadatas = metadatas[0]
                
            for i, doc in enumerate(docs):
                if i < len(metadatas):
                    article = {
                        'text': doc,
                        'title': metadatas[i].get('title', 'No Title'),
                        'summary': metadatas[i].get('summary', ''),
                        'source': metadatas[i].get('source', 'Unknown'),
                        'stock_symbol': symbol
                    }
                    # Only include articles with reference summaries
                    if article['summary'].strip():
                        articles.append(article)
    
    return articles

# Model initialization

### init_models



In [15]:
def init_models():
    """Initialize both baseline and fine-tuned models."""
    # Baseline model
    try:
        baseline_tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL)
        baseline_model = AutoModelForSeq2SeqLM.from_pretrained(BASELINE_MODEL)
        
        # Simplified interface through pipeline
        baseline_pipeline = pipeline(
            "summarization", 
            model=BASELINE_MODEL,
            device=-1  # CPU
        )
    except Exception as e:
        print(f"Error initializing baseline model: {str(e)}")
        baseline_tokenizer, baseline_model, baseline_pipeline = None, None, None
    
    # Fine-tuned model
    try:
        if os.path.exists(FINETUNED_MODEL_DIR):
            finetuned_tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_DIR)
            finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(FINETUNED_MODEL_DIR)
        else:
            print(f"Fine-tuned model not found at {FINETUNED_MODEL_DIR}")
            finetuned_tokenizer, finetuned_model = None, None
    except Exception as e:
        print(f"Error initializing fine-tuned model: {str(e)}")
        finetuned_tokenizer, finetuned_model = None, None
    
    return {
        'baseline': {
            'tokenizer': baseline_tokenizer,
            'model': baseline_model,
            'pipeline': baseline_pipeline
        },
        'finetuned': {
            'tokenizer': finetuned_tokenizer,
            'model': finetuned_model
        }
    }

# Generate summaries

### generate_summaries



In [16]:
def generate_summaries(articles, models):
    """Generate summaries using both models for all test articles."""
    results = []
    
    for article in tqdm(articles, desc="Generating summaries"):
        text = article['text']
        
        # Truncate to max 4000 chars to avoid tokenization issues
        if len(text) > 4000:
            text = text[:4000]
        
        result = {
            'title': article['title'],
            'original_text': text,
            'reference_summary': article['summary'],
            'stock_symbol': article['stock_symbol'],
            'source': article['source']
        }
        
        # Baseline model summary
        try:
            if models['baseline']['pipeline']:
                baseline_output = models['baseline']['pipeline'](
                    text,
                    max_length=150,
                    min_length=40,
                    do_sample=False
                )
                result['baseline_summary'] = baseline_output[0]['summary_text']
            else:
                result['baseline_summary'] = "Model not available"
        except Exception as e:
            print(f"Error with baseline summary: {str(e)}")
            result['baseline_summary'] = f"Error: {str(e)}"
        
        # Fine-tuned model summary
        try:
            if models['finetuned']['model'] and models['finetuned']['tokenizer']:
                tokenizer = models['finetuned']['tokenizer']
                model = models['finetuned']['model']
                
                inputs = tokenizer(text, return_tensors="pt", max_length=1024, 
                                  truncation=True, padding="max_length")
                
                summary_ids = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_length=150,
                    min_length=40,
                    num_beams=4,
                    length_penalty=2.0,
                    early_stopping=True,
                    no_repeat_ngram_size=3
                )
                
                result['finetuned_summary'] = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            else:
                result['finetuned_summary'] = "Model not available"
        except Exception as e:
            print(f"Error with fine-tuned summary: {str(e)}")
            result['finetuned_summary'] = f"Error: {str(e)}"
        
        results.append(result)
    
    return results

# Standard metric evaluation (ROUGE, BLEU)

### evaluate_standard_metrics



In [17]:
def evaluate_standard_metrics(results):
    """Evaluate results using standard metrics like ROUGE and BLEU."""
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    metrics = {
        'baseline': {
            'rouge1': [], 'rouge2': [], 'rougeL': [], 'bleu': []
        },
        'finetuned': {
            'rouge1': [], 'rouge2': [], 'rougeL': [], 'bleu': []
        }
    }
    
    for result in results:
        reference = result['reference_summary']
        
        # Skip if reference summary is not available
        if not reference or reference == "No summary available":
            continue
        
        # Tokenize reference for BLEU
        reference_tokens = nltk.word_tokenize(reference.lower())
        
        # Baseline metrics
        if 'baseline_summary' in result and not result['baseline_summary'].startswith('Error'):
            baseline = result['baseline_summary']
            
            # ROUGE scores
            rouge_scores = rouge.score(reference, baseline)
            metrics['baseline']['rouge1'].append(rouge_scores['rouge1'].fmeasure)
            metrics['baseline']['rouge2'].append(rouge_scores['rouge2'].fmeasure)
            metrics['baseline']['rougeL'].append(rouge_scores['rougeL'].fmeasure)
            
            # BLEU score
            baseline_tokens = nltk.word_tokenize(baseline.lower())
            bleu = sentence_bleu([reference_tokens], baseline_tokens)
            metrics['baseline']['bleu'].append(bleu)
        
        # Fine-tuned metrics
        if 'finetuned_summary' in result and not result['finetuned_summary'].startswith('Error'):
            finetuned = result['finetuned_summary']
            
            # ROUGE scores
            rouge_scores = rouge.score(reference, finetuned)
            metrics['finetuned']['rouge1'].append(rouge_scores['rouge1'].fmeasure)
            metrics['finetuned']['rouge2'].append(rouge_scores['rouge2'].fmeasure)
            metrics['finetuned']['rougeL'].append(rouge_scores['rougeL'].fmeasure)
            
            # BLEU score
            finetuned_tokens = nltk.word_tokenize(finetuned.lower())
            bleu = sentence_bleu([reference_tokens], finetuned_tokens)
            metrics['finetuned']['bleu'].append(bleu)
    
    # Calculate averages
    avg_metrics = {
        'baseline': {
            'rouge1': np.mean(metrics['baseline']['rouge1']) if metrics['baseline']['rouge1'] else 0,
            'rouge2': np.mean(metrics['baseline']['rouge2']) if metrics['baseline']['rouge2'] else 0,
            'rougeL': np.mean(metrics['baseline']['rougeL']) if metrics['baseline']['rougeL'] else 0,
            'bleu': np.mean(metrics['baseline']['bleu']) if metrics['baseline']['bleu'] else 0
        },
        'finetuned': {
            'rouge1': np.mean(metrics['finetuned']['rouge1']) if metrics['finetuned']['rouge1'] else 0,
            'rouge2': np.mean(metrics['finetuned']['rouge2']) if metrics['finetuned']['rouge2'] else 0,
            'rougeL': np.mean(metrics['finetuned']['rougeL']) if metrics['finetuned']['rougeL'] else 0,
            'bleu': np.mean(metrics['finetuned']['bleu']) if metrics['finetuned']['bleu'] else 0
        }
    }
    
    return avg_metrics, metrics

# Factual consistency evaluation

### evaluate_factual_consistency



In [23]:
# Factual consistency evaluation
def evaluate_factual_consistency(results):
    """Evaluate factual consistency between source text and summaries."""
    def extract_entities(text):
        doc = nlp(text)
        entities = set()
        for ent in doc.ents:
            if ent.label_ in ['ORG', 'PERSON', 'GPE', 'MONEY', 'PERCENT', 'DATE', 'CARDINAL']:
                entities.add((ent.text, ent.label_))
        return entities
    
    def entity_overlap_score(source_entities, summary_entities):
        if not source_entities:
            return 0
        
        # Only consider entities mentioned in source
        summary_matches = set()
        for summary_ent, label in summary_entities:
            # Look for exact or partial matches
            for source_ent, source_label in source_entities:
                if summary_ent in source_ent or source_ent in summary_ent:
                    summary_matches.add((summary_ent, label))
                    break
        
        precision = len(summary_matches) / len(summary_entities) if summary_entities else 0
        recall = len(summary_matches) / len(source_entities) if source_entities else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return f1
    
    # Semantic similarity using sentence embeddings
    def semantic_similarity(text1, text2):
        emb1 = sentence_model.encode(text1, convert_to_tensor=True)
        emb2 = sentence_model.encode(text2, convert_to_tensor=True)
        similarity = torch.nn.functional.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0))
        return similarity.item()
    
    factual_metrics = {
        'baseline': {
            'entity_overlap': [],
            'semantic_similarity': [],
            'hallucinations': []
        },
        'finetuned': {
            'entity_overlap': [],
            'semantic_similarity': [],
            'hallucinations': []
        }
    }
    
    for result in tqdm(results, desc="Evaluating factual consistency"):
        text = result['original_text']
        source_entities = extract_entities(text)
        
        # Baseline evaluation
        if 'baseline_summary' in result and not result['baseline_summary'].startswith('Error'):
            baseline = result['baseline_summary']
            baseline_entities = extract_entities(baseline)
            
            # Entity overlap
            overlap = entity_overlap_score(source_entities, baseline_entities)
            factual_metrics['baseline']['entity_overlap'].append(overlap)
            
            # Semantic similarity
            sim = semantic_similarity(text, baseline)
            factual_metrics['baseline']['semantic_similarity'].append(sim)
            
            # Potential hallucinations - entities in summary not in source
            hallucinations = []
            for ent, label in baseline_entities:
                if not any(ent in src_ent or src_ent in ent for src_ent, _ in source_entities):
                    hallucinations.append((ent, label))
            
            factual_metrics['baseline']['hallucinations'].append(len(hallucinations) / len(baseline_entities) if baseline_entities else 0)
        
        # Fine-tuned evaluation
        if 'finetuned_summary' in result and not result['finetuned_summary'].startswith('Error'):
            finetuned = result['finetuned_summary']
            finetuned_entities = extract_entities(finetuned)
            
            # Entity overlap
            overlap = entity_overlap_score(source_entities, finetuned_entities)
            factual_metrics['finetuned']['entity_overlap'].append(overlap)
            
            # Semantic similarity
            sim = semantic_similarity(text, finetuned)
            factual_metrics['finetuned']['semantic_similarity'].append(sim)
            
            # Potential hallucinations - entities in summary not in source
            hallucinations = []
            for ent, label in finetuned_entities:
                if not any(ent in src_ent or src_ent in ent for src_ent, _ in source_entities):
                    hallucinations.append((ent, label))
            
            factual_metrics['finetuned']['hallucinations'].append(len(hallucinations) / len(finetuned_entities) if finetuned_entities else 0)
    
    # Calculate averages
    avg_factual_metrics = {
        'baseline': {
            'entity_overlap': np.mean(factual_metrics['baseline']['entity_overlap']) if factual_metrics['baseline']['entity_overlap'] else 0,
            'semantic_similarity': np.mean(factual_metrics['baseline']['semantic_similarity']) if factual_metrics['baseline']['semantic_similarity'] else 0,
            'hallucination_rate': np.mean(factual_metrics['baseline']['hallucinations']) if factual_metrics['baseline']['hallucinations'] else 0
        },
        'finetuned': {
            'entity_overlap': np.mean(factual_metrics['finetuned']['entity_overlap']) if factual_metrics['finetuned']['entity_overlap'] else 0,
            'semantic_similarity': np.mean(factual_metrics['finetuned']['semantic_similarity']) if factual_metrics['finetuned']['semantic_similarity'] else 0,
            'hallucination_rate': np.mean(factual_metrics['finetuned']['hallucinations']) if factual_metrics['finetuned']['hallucinations'] else 0
        }
    }
    
    return avg_factual_metrics, factual_metrics

### semantic_similarity

### visualize_standard_metrics



In [24]:
def visualize_standard_metrics(avg_metrics):
    """Visualize standard metric comparisons between models."""
    metrics = ['rouge1', 'rouge2', 'rougeL', 'bleu']
    baseline_scores = [avg_metrics['baseline'][m] for m in metrics]
    finetuned_scores = [avg_metrics['finetuned'][m] for m in metrics]
    
    plt.figure(figsize=(10, 6))
    
    x = np.arange(len(metrics))
    width = 0.35
    
    plt.bar(x - width/2, baseline_scores, width, label='Baseline')
    plt.bar(x + width/2, finetuned_scores, width, label='Fine-tuned')
    
    plt.xticks(x, metrics)
    plt.ylabel('Score')
    plt.title('Standard Metrics Comparison')
    plt.legend()
    plt.ylim(0, 1)
    
    plt.savefig(os.path.join(RESULTS_DIR, 'standard_metrics.png'))
    plt.close()

### visualize_factual_metrics



In [25]:
def visualize_factual_metrics(avg_factual_metrics):
    """Visualize factual metrics comparisons between models."""
    metrics = ['entity_overlap', 'semantic_similarity', 'hallucination_rate']
    labels = ['Entity Overlap', 'Semantic Similarity', 'Hallucination Rate']
    
    baseline_scores = [avg_factual_metrics['baseline'][m] for m in metrics]
    finetuned_scores = [avg_factual_metrics['finetuned'][m] for m in metrics]
    
    plt.figure(figsize=(10, 6))
    
    x = np.arange(len(metrics))
    width = 0.35
    
    plt.bar(x - width/2, baseline_scores, width, label='Baseline')
    plt.bar(x + width/2, finetuned_scores, width, label='Fine-tuned')
    
    plt.xticks(x, labels)
    plt.ylabel('Score')
    plt.title('Factual Metrics Comparison')
    plt.legend()
    plt.ylim(0, 1)
    
    plt.savefig(os.path.join(RESULTS_DIR, 'factual_metrics.png'))
    plt.close()

### visualize_improvement



In [26]:
def visualize_improvement(avg_metrics, avg_factual_metrics):
    """Visualize the improvement gained by fine-tuning."""
    # Calculate improvement percentages
    improvements = {
        'Standard Metrics': {
            'rouge1': (avg_metrics['finetuned']['rouge1'] - avg_metrics['baseline']['rouge1']) / max(avg_metrics['baseline']['rouge1'], 0.001) * 100,
            'rouge2': (avg_metrics['finetuned']['rouge2'] - avg_metrics['baseline']['rouge2']) / max(avg_metrics['baseline']['rouge2'], 0.001) * 100,
            'rougeL': (avg_metrics['finetuned']['rougeL'] - avg_metrics['baseline']['rougeL']) / max(avg_metrics['baseline']['rougeL'], 0.001) * 100,
            'bleu': (avg_metrics['finetuned']['bleu'] - avg_metrics['baseline']['bleu']) / max(avg_metrics['baseline']['bleu'], 0.001) * 100
        },
        'Factual Metrics': {
            'entity_overlap': (avg_factual_metrics['finetuned']['entity_overlap'] - avg_factual_metrics['baseline']['entity_overlap']) / max(avg_factual_metrics['baseline']['entity_overlap'], 0.001) * 100,
            'semantic_similarity': (avg_factual_metrics['finetuned']['semantic_similarity'] - avg_factual_metrics['baseline']['semantic_similarity']) / max(avg_factual_metrics['baseline']['semantic_similarity'], 0.001) * 100,
            'hallucination_rate': (avg_factual_metrics['baseline']['hallucination_rate'] - avg_factual_metrics['finetuned']['hallucination_rate']) / max(avg_factual_metrics['baseline']['hallucination_rate'], 0.001) * 100  # Reversed as lower is better
        }
    }
    
    # Create a combined plot
    plt.figure(figsize=(12, 8))
    
    # Plot standard metrics improvements
    metrics = list(improvements['Standard Metrics'].keys())
    values = list(improvements['Standard Metrics'].values())
    colors = ['blue'] * len(metrics)
    
    # Add factual metrics
    metrics.extend(list(improvements['Factual Metrics'].keys()))
    values.extend(list(improvements['Factual Metrics'].values()))
    colors.extend(['green'] * len(improvements['Factual Metrics']))
    
    # Create bars
    bars = plt.bar(metrics, values, color=colors)
    
    # Add labels and title
    plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
    plt.ylabel('Improvement (%)')
    plt.title('Fine-tuning Impact: Percentage Improvement Over Baseline')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Add values on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + (5 if height > 0 else -15),
                 f'{height:.1f}%', ha='center', va='bottom')
    
    plt.savefig(os.path.join(RESULTS_DIR, 'improvement.png'))
    plt.close()

# Main function

### main



In [27]:
def main():
    """Run the complete evaluation pipeline."""
    print("Starting summarization model evaluation...")
    
    # Initialize vector database connection
    vector_db = init_vector_db()
    
    # Get test articles
    articles = get_test_articles(vector_db, n_articles=30)  # Adjust number as needed
    print(f"Retrieved {len(articles)} articles for evaluation")
    
    # Initialize models
    models = init_models()
    
    # Generate summaries for all articles
    results = generate_summaries(articles, models)
    
    # Save raw results
    with open(os.path.join(RESULTS_DIR, 'raw_results.json'), 'w') as f:
        # Convert to simpler dictionary for JSON serialization
        serializable_results = []
        for r in results:
            serializable_results.append({k: str(v) for k, v in r.items()})
        json.dump(serializable_results, f, indent=2)
    
    # Evaluate with standard metrics
    avg_metrics, detailed_metrics = evaluate_standard_metrics(results)
    
    # Evaluate factual consistency
    avg_factual_metrics, detailed_factual_metrics = evaluate_factual_consistency(results)
    
    # Create visualizations
    visualize_standard_metrics(avg_metrics)
    visualize_factual_metrics(avg_factual_metrics)
    visualize_improvement(avg_metrics, avg_factual_metrics)
    
    # Save metrics to CSV
    metrics_df = pd.DataFrame({
        'Metric': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BLEU', 'Entity Overlap', 'Semantic Similarity', 'Hallucination Rate'],
        'Baseline': [
            avg_metrics['baseline']['rouge1'],
            avg_metrics['baseline']['rouge2'],
            avg_metrics['baseline']['rougeL'],
            avg_metrics['baseline']['bleu'],
            avg_factual_metrics['baseline']['entity_overlap'],
            avg_factual_metrics['baseline']['semantic_similarity'],
            avg_factual_metrics['baseline']['hallucination_rate']
        ],
        'Fine-tuned': [
            avg_metrics['finetuned']['rouge1'],
            avg_metrics['finetuned']['rouge2'],
            avg_metrics['finetuned']['rougeL'],
            avg_metrics['finetuned']['bleu'],
            avg_factual_metrics['finetuned']['entity_overlap'],
            avg_factual_metrics['finetuned']['semantic_similarity'],
            avg_factual_metrics['finetuned']['hallucination_rate']
        ]
    })
    
    # Calculate improvement
    metrics_df['Improvement (%)'] = [
        (metrics_df['Fine-tuned'][0] - metrics_df['Baseline'][0]) / max(metrics_df['Baseline'][0], 0.001) * 100,
        (metrics_df['Fine-tuned'][1] - metrics_df['Baseline'][1]) / max(metrics_df['Baseline'][1], 0.001) * 100,
        (metrics_df['Fine-tuned'][2] - metrics_df['Baseline'][2]) / max(metrics_df['Baseline'][2], 0.001) * 100,
        (metrics_df['Fine-tuned'][3] - metrics_df['Baseline'][3]) / max(metrics_df['Baseline'][3], 0.001) * 100,
        (metrics_df['Fine-tuned'][4] - metrics_df['Baseline'][4]) / max(metrics_df['Baseline'][4], 0.001) * 100,
        (metrics_df['Fine-tuned'][5] - metrics_df['Baseline'][5]) / max(metrics_df['Baseline'][5], 0.001) * 100,
        (metrics_df['Baseline'][6] - metrics_df['Fine-tuned'][6]) / max(metrics_df['Baseline'][6], 0.001) * 100,  # Reversed for hallucination
    ]
    
    metrics_df.to_csv(os.path.join(RESULTS_DIR, 'evaluation_metrics.csv'), index=False)
    
    print("Evaluation complete! Results saved to:", RESULTS_DIR)
    print("\nSummary of improvements:")
    print(metrics_df[['Metric', 'Improvement (%)']])

if __name__ == "__main__":
    main()

Starting summarization model evaluation...
Initializing ChromaDB with persistence directory: ../data/chroma_db
Loaded existing collection 'financial_articles'
Collection 'financial_articles' contains 400 documents
Found 6 results. Showing top 3:

Result #1 (Similarity: 0.1246)
Title: Inquiry Into Apple's Competitor Dynamics In Technology Hardware, Storage & Peripherals Industry
Source: Benzinga
Date: 2025-05-16 15:00:43
Chunk: 1 of 3
Link: https://www.benzinga.com/insights/news/25/05/45470918/inquiry-into-apples-competitor-dynamics-in-technology-hardware-storage-amp-peripherals-industry
Creator: Benzinga Insights
Summary: In this article, we will perform an extensive industry comparison, evaluating Apple AAPL in relation to its major competitors in the Technology Hardware, Storage & Peripherals industry.
The Return on Equity (ROE) of 37.11% is 31.03% above the industry average, highlighting efficient use of equity to generate profits.
The company's revenue growth of 5.08% is significan

/Users/giannisalexandrou/.local/share/virtualenvs/trading-agent-Iwjhydeb/lib/python3.12/site-packages/transformers/models/bart/configuration_bart.py:179: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


Error initializing fine-tuned model: `early_stopping` must be a boolean or 'never', but is None.


Generating summaries: 100%|██████████| 30/30 [02:17<00:00,  4.59s/it]
/Users/giannisalexandrou/.local/share/virtualenvs/trading-agent-Iwjhydeb/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/Users/giannisalexandrou/.local/share/virtualenvs/trading-agent-Iwjhydeb/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/Users/giannisalexandrou/.local/share/virtualenvs/trading-agent-Iwjhydeb/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: 

Evaluation complete! Results saved to: ../results/model_evaluation

Summary of improvements:
                Metric  Improvement (%)
0              ROUGE-1       -98.575155
1              ROUGE-2      -100.000000
2              ROUGE-L       -98.610301
3                 BLEU      -100.000000
4       Entity Overlap      -100.000000
5  Semantic Similarity       -94.393345
6   Hallucination Rate       100.000000


## Main Execution

In [ ]:
main()